# DATALUS OAA Compliance: Empirical Privacy & Utility Audit

This notebook demonstrates the **Autonomous Audit Orchestrator (OAA)**. 
We evaluate the generated synthetic data against rigorous privacy and utility benchmarks to prove its safety for public release 
under data protection regulations (e.g., LGPD/GDPR).

The audit includes **Membership Inference Attack (MIA)** simulations and **Distance to Closest Record (DCR)** checks for memorization.

## 1. Environment Setup & Data Verification
Detecting environment and verifying that real and synthetic datasets are ready for audit. 
If data is missing, we trigger a fallback generation process.

In [ ]:
import os
import sys
from pathlib import Path
import polars as pl
import numpy as np

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_audit'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_audit'
    return './datalus_audit'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

# Install dependencies
if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    !git clone https://github.com/emanuellcs/datalus.git || true
    %cd datalus
    !python -m pip install -e '.[dev]' pysus polars matplotlib seaborn scikit-learn
    sys.path.append(os.getcwd())

real_path = Path(f'{WORKING_DIR}/processed.parquet')
synth_path = Path(f'{WORKING_DIR}/synthetic.parquet')
schema_path = Path(f'{WORKING_DIR}/schema_config.json')

if not (real_path.exists() and synth_path.exists()):
    print('Data not found. Generating fallback dummy datasets for audit demo...')
    from pysus.api import sih
    df = pl.from_pandas(sih(state='SP', year=2024, month=[1]))
    df = df.with_columns(pl.col('MORTE').cast(pl.Int64).fill_null(0)).head(2000)
    df.write_parquet(f'{WORKING_DIR}/raw_sample.parquet')
    
    !datalus ingest {WORKING_DIR}/raw_sample.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE
    !datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 1 --batch-size 256
    !datalus sample {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet --n-records 2000
else:
    print('Verified: Audit datasets are ready.')


## 2. Executing the Autonomous Audit Orchestrator (OAA)
We run the `datalus audit` command. For this demonstration, we use a limited subset of rows to ensure fast execution. 
The `--mia-mode release` flag triggers the full shadow-model attack simulation.

In [ ]:
!datalus audit {WORKING_DIR}/processed.parquet {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/schema_config.json {WORKING_DIR}/audit_report.json --target-column MORTE --mia-mode release

## 3. Parsing and Interpreting the Audit Results
Loading the JSON report and extracting the high-level Pass/Fail verdict based on privacy and utility thresholds.

In [ ]:
import json
with open(f'{WORKING_DIR}/audit_report.json', 'r') as f:
    report = json.load(f)

verdict = "PASSED" if report.get('overall_verdict', False) else "FAILED"
print(f"--- AUDIT VERDICT: {verdict} ---")
print(f"Privacy Score: {report.get('privacy', {}).get('mia_roc_auc', 'N/A')}")
print(f"Utility MLE Ratio: {report.get('utility', {}).get('mle_ratio', 'N/A')}")


## 4. Privacy Validation: Distance to Closest Record (DCR)
DCR measures if the model has memorized specific real records. We check the 5th percentile distance against the alert threshold.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'privacy' in report and 'dcr_stats' in report['privacy']:
    dcr = report['privacy']['dcr_stats']
    p5_dcr = dcr.get('p5', 0)
    threshold = 0.05 # Example threshold
    
    plt.figure(figsize=(10, 5))
    colors = ['green' if p5_dcr > threshold else 'red']
    plt.bar(['5th Percentile DCR'], [p5_dcr], color=colors)
    plt.axhline(y=threshold, color='black', linestyle='--', label='Security Threshold')
    plt.title('Privacy Check: Distance to Closest Record (Memorization Risk)')
    plt.ylabel('Normalized Distance')
    plt.legend()
    plt.show()


## 5. Privacy Validation: Shadow-MIA Attack AUC
An AUC close to 0.50 indicates that the attacker cannot distinguish between members and non-members of the training set.

In [ ]:
if 'privacy' in report and 'mia_roc_auc' in report['privacy']:
    auc = report['privacy']['mia_roc_auc']
    
    plt.figure(figsize=(10, 5))
    plt.bar(['MIA ROC-AUC'], [auc], color='orange')
    plt.axhline(y=0.5, color='blue', linestyle='--', label='Random Guess (Perfect Privacy)')
    plt.axhline(y=0.6, color='red', linestyle='--', label='Alert Threshold')
    plt.ylim(0.4, 1.0)
    plt.title('Privacy Check: Membership Inference Attack Resistance')
    plt.ylabel('ROC-AUC Score')
    plt.legend()
    plt.show()


## 6. Utility Validation: TSTR vs TRTR Machine Learning Efficacy
The MLE ratio proves that models trained on synthetic data perform nearly as well as those trained on real data.

In [ ]:
if 'utility' in report and 'mle_scores' in report['utility']:
    mle = report['utility']['mle_scores']
    labels = ['TSTR', 'TRTR']
    scores = [mle.get('tstr', 0), mle.get('trtr', 0)]
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x=labels, y=scores, palette='viridis')
    plt.title(f"Utility Check: MLE Ratio ({report['utility'].get('mle_ratio', 0):.2f})")
    plt.ylabel('F1 / Accuracy Score')
    plt.show()
